Script Overview:
1. Excluded words are combined with spaCys default stop words
2. Docs are tokenized from line 6 (excluding date and title) and words are excluded
3. Words with a frequency of one or two are removed and terms occuring in more than 95% of the docs are removed
4. The tokenized text is prepared for topic modeling and converted to a BOW
5. Hyperparameter optimization is performed to find the best parameters for the LDA model
6. LDA model is run with best parameters
7. Topics are visualized 
8. Top 10 words per topic are printed
9. Uncertainty terms calculated per doc
10. Number of docs per topic are calculated through the probability distribution given for each document from LDA (e.g if 50% of doc 1 is topic 2 then topic 2 gets given 0.5)
11. Number of uncertainty terms per topic is calculated the same way (e.g. doc 1 has 5 uncertainty terms, 50% of doc 1 is topic 2 so topic 2 gets assigned 2.5 words)
12. Chi square goodness of fit assumption check, if met test is performed if not permutation test is performed. 

1. Excluded words are combined with spaCys default stop words

In [ ]:
# combine excluded words with spacys default
excluded_words = ["the", "in", "to", "as","an", "of", "and", "from", "a", "by", "for", "their", "than", "were", "this", "or", "is", 
                  "which","was", "have", "with","  ","those", "who", "on", "had", "that", "but", "into", "are", "also", "$",
                  "article","n05b", "n06b", "at", "Australia", "ABS", "/n", "²⁰", "¹²", "¹³", "¹¹", "¹⁰", "¹⁴", "¹⁵", "¹⁶", "¹⁷","¹⁹", "²²", "²³", "²¹", "australia",
                  "be", "these", "other", "abs", "has", "there", "data", "not", "more", "due", "over", "people", "age", "time"]
excluded_words.extend(nlp.Defaults.stop_words)
excluded_words = list(set(excluded_words))

2. Docs are tokenized from line 6 (excluding date and title) and words are excluded

In [4]:
import os
import re
from datetime import datetime
import spacy
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer

nlp = spacy.load("en_core_web_sm")


excluded_tokens = set()
def tokenize_text(file):
    with open(file, "r") as f:
        lines = f.readlines()[6:]  
        
    text = " ".join(lines)

    # Create doc with tokens
    doc = nlp(text)

    # Remove punctuation, excluded words, and non-alphabetic tokens
    words = []
    for token in doc:
        if token.is_alpha and token.text.lower() not in excluded_words: 
            words.append(token.text.lower())
        else:
            excluded_tokens.add(token.text.lower())

    return " ".join(words)


directory = "filtered_a"

docs = []
filenames = []
for file in os.listdir(directory):
    file_path = os.path.join(directory, file)
    token_text = tokenize_text(file_path)
    docs.append(token_text)
    filenames.append(file)

vectorizer = CountVectorizer(stop_words = excluded_words)

# outputs a sparse matrix
ABS_matrix = vectorizer.fit_transform(docs)

/home/iwag@cbsp.nl/.local/lib/python3.10/site-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ll', 've'] not in stop_words.
  warnings.warn(


3. Words with a frequency of one or two are removed and terms occuring in more than 95% of the docs are removed

In [6]:
import numpy as np
import pandas as pd
abs = pd.DataFrame(ABS_matrix.toarray(), index = filenames, columns = vectorizer.get_feature_names_out())

In [7]:
# removing cols with 1 & 2 terms
abs_t_1 = abs.columns[abs.sum(axis = 0)==1]
abs = abs.drop(columns = abs_t_1)
abs_t_2 = abs.columns[abs.sum(axis = 0)==2]
abs = abs.drop(columns = abs_t_2)
term_counts = (abs > 0).sum(axis = 0)
# getting terms that occur in more than 95% of docs
terms_95 = abs.columns[term_counts > 0.95*len(abs)]

abs = abs.drop(columns= terms_95)

4. The tokenized text is prepared for topic modeling and converted to a BOW

In [9]:
# load libraries for topic modeling
import gensim
from scipy.sparse import csr_matrix
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

In [10]:
# convert dictionary into BOW format for Topic Modeling
dictionary = Dictionary([[word] for word in abs.columns])
bow = []
for idx, row in abs.iterrows():
    doc_bow = [(dictionary.token2id[word], count) for word, count in row.items() if count > 0]
    bow.append(doc_bow)

5. Hyperparameter optimization is performed to find the best parameters for the LDA model

In [ ]:
import numpy as np

num_topics_range = [4,6,8,10]
alpha =["auto","symmetric", "asymmetric", 0.1,0.3,0.5, 0.8]
eta = ["auto","symmetric", 0.01, 0.03, 0.05, 0.08, 0.1, 0.5, 0.7, 0.9]


def coherence_score_fnc(model, texts, dictionary):
    cv_model = CoherenceModel(model = model, texts = texts, dictionary = dictionary, coherence= "c_v")
    cv_score = cv_model.get_coherence()
    return cv_score

texts = [doc.split() for doc in docs]

best_coherence = -1
best_model = None
best_params = None


# set a seed for reproducibility
seed = 42

scores = []

for num_topics in num_topics_range:
    for a in alpha:
        for e in eta:
            print(f"Model with topics:{num_topics}, alpha:{a}, eta:{e}")

            lda_model = gensim.models.LdaModel(bow, num_topics= num_topics, id2word = dictionary, passes= 10, alpha = a, eta = e, random_state= seed)

            coherence_score = coherence_score_fnc(model = lda_model, texts = texts, dictionary= dictionary)
            print(f"Coherence Score: {coherence_score}")

            scores.append((coherence_score, num_topics, a ,e))

        

scores.sort(reverse = True, key = lambda x:x[0])
for i in range(min(5, len(scores))):
    print(f"Position {i + 1}: Coherence Score = {scores[i][0]}, Parameters: num_topics = {scores[i][1]}, alpha = {scores[i][2]}, eta = {scores[i][3]}")


6. LDA model is run with best parameters

In [11]:
# LDA Model with best parameters:

lda = gensim.models.LdaModel(
    bow, num_topics=4, id2word=dictionary, passes=10, alpha= 0.8, eta=0.01, random_state= 42
)

coherence_model = CoherenceModel(model= lda, texts = [doc.split() for doc in docs] , dictionary= dictionary,
                              coherence = "c_v")
with np.errstate(invalid= "ignore"): 
    c_cv = coherence_model.get_coherence()  

#print measure
c_cv 

0.5518530583080556

7. Topics are visualized 

In [ ]:
# visualise results with LDAvis
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis
visual = gensimvis.prepare(lda, bow, dictionary)

pyLDAvis.display(visual)

8. Top 10 words per topic are printed

In [ ]:
# Extract top 10 words and their weights for each topic
for i, topic in enumerate(lda.show_topics(num_topics=4, num_words=10, formatted=False)):
    print(f"Topic {i+1}:")
    for word, weight in topic[1]:
        print(f"  {word}: {weight}")
 


Topic 1:
  covid: 0.05093624070286751
  deaths: 0.04692118987441063
  death: 0.03206059709191322
  islander: 0.015569071285426617
  torres: 0.015507036820054054
  aboriginal: 0.015454067848622799
  strait: 0.015312552452087402
  mortality: 0.01512086857110262
  conditions: 0.011356934905052185
  cause: 0.011315284296870232
Topic 2:
  labour: 0.015437526628375053
  estimates: 0.010589422658085823
  jobs: 0.008400565944612026
  census: 0.00794241577386856
  hours: 0.007858309894800186
  changes: 0.007065645884722471
  population: 0.006626471411436796
  industry: 0.006608792580664158
  worked: 0.00582292303442955
  businesses: 0.00571786193177104
Topic 3:
  quarter: 0.012618609704077244
  services: 0.009849196299910545
  cpi: 0.008126527070999146
  increased: 0.008040506392717361
  cent: 0.00795009545981884
  income: 0.007893453352153301
  price: 0.0074296919628977776
  billion: 0.0073885079473257065
  government: 0.00679079070687294
  covid: 0.0062857638113200665
Topic 4:
  years: 0.0134

9. Uncertainty terms calculated per doc

In [10]:
# first lets start by tokenizing the text and looking for uncertainty 
import os
import spacy
from nltk.util import ngrams
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns



nlp = spacy.load("en_core_web_sm")



uncertainty_terms = set([
    "always", "certain", "chance", "common", "doubtful", "expected","frequently","generally", "likely", 
    "impossible","never","inconclusive","often",
    "possible", "predicatable", "probable","rarely", "seldom", "slightly", "slight", "sometimes", 
    "uncertain", "uncommon","usually", "unlikely"
])


# tokenization fnc
excluded_tokens = set()
def tokenize_text(file_path):
    with open(file_path, "r") as f:
        lines = f.readlines()[6:]
        text = "".join(lines)

    doc = nlp(text)
    words = []
    return doc
    

# uncertainty fnc
def find_uncertainty_terms(doc_tokens):
    found_terms = []
    found_terms += [w.text.lower() for w in doc_tokens if w.text.lower() in uncertainty_terms]
    return found_terms


# Directory with files
directory = "filtered_a"  


tokenize_docs = []
filenames = []


for file in os.listdir(directory):
    file_path = os.path.join(directory, file)
    doc = tokenize_text(file_path)
    filenames.append(file)
    tokenize_docs.append(doc)
    

10. Number of docs per topic are calculated through the probability distribution given for each document from LDA (e.g if 50% of doc 1 is topic 2 then topic 2 gets given 0.5)
11. Number of uncertainty terms per topic is calculated the same way (e.g. doc 1 has 5 uncertainty terms, 50% of doc 1 is topic 2 so topic 2 gets assigned 2.5 words)

In [ ]:
# Initialize topic uncertainty counts with weights
topic_u_weighted_counts = {i:0 for i in range(lda.num_topics)}
topic_doc_counts = {i:0 for i in range(lda.num_topics)}

for idx, content in enumerate(bow):
    doc_topic_dist = lda.get_document_topics(content)
    doc_tok = tokenize_docs[idx]
 
    found_terms = find_uncertainty_terms(doc_tok)
    num_uncertainty_terms = len(found_terms)
    
    
    for topic, prob in doc_topic_dist:
        topic_doc_counts[topic] += prob
        weighted_uncertainty = num_uncertainty_terms * prob
        topic_u_weighted_counts[topic] += weighted_uncertainty


topic_u_weighted_counts
topic_doc_counts

{0: 641.2986808372661,
 1: 338.3762822980061,
 2: 158.24214638303965,
 3: 618.9632500195876}

In [ ]:
data = {"Topic": list(topic_u_weighted_counts.keys()),
        "Uncertainty Terms": list(topic_u_weighted_counts.values()),
        "Documents": list(topic_doc_counts.values())
        }


df = pd.DataFrame(data)
df["Relative Freq"] = df["Uncertainty Terms"]/df["Documents"]

12. Chi square goodness of fit assumption check, if met test is performed if not permutation test is performed. 

In [ ]:
from scipy.stats import chisquare 
# can perform chi square as E > 5
obs = df["Relative Freq"]
exp = [sum(obs)/len(obs)]*len(obs)

chi2, p_value = chisquare(obs, exp)

In [17]:
chi2, p_value

(11.667764752981544, 0.008612397969887234)